# NB05 — Training Only (T4 Quick Test)

Minimal notebook: clone, install, copy data, **train 3 epochs**.
No simulation, no drift, no Airflow. Just GPU training.

**Purpose:** Verify training completes on T4 before running full 40-day sim.
**Expected:** ~40-50 min on T4 with `batch_size=1024`.

**Key fixes (v7C):**
- GradScaler `init_scale=1024` for float16 stability
- NaN predictions filtered before metric computation
- Threshold selection now ignores non-finite probabilities instead of crashing
- T4/V100 float16 runs use `batch_size=1024` for stability, not `2048`
- If non-finite validation probabilities become too frequent, training now fails explicitly instead of pretending the run is healthy

In [ ]:
import os, shutil, subprocess

REPO = "https://github.com/AIML-Engineering-Lab/053_dram_memory_yield_mlops.git"
DIR = "/content/053_dram_memory_yield_mlops"

try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = REPO.replace("https://", f"https://{tok}@") if tok else REPO

if os.path.exists(DIR):
    shutil.rmtree(DIR)
r = subprocess.run(["git", "clone", url, DIR], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"Clone failed: {r.stderr}")
os.chdir(DIR)

# Verify ALL fixes are present in cloned code
with open('src/train.py') as f:
    train_src = f.read()
with open('src/model.py') as f:
    model_src = f.read()
checks = [
    ('init_scale=1024' in train_src, 'GradScaler init_scale=1024'),
    ('sanitize_probabilities' in train_src, 'NaN-safe probability sanitization'),
    ('capping to 1024 to avoid T4/V100 AMP instability' in train_src, 'T4 batch-size stability cap'),
    ('np.isnan(preds)' in train_src, 'NaN-safe prediction collection'),
    ('No finite probabilities available for threshold selection' in model_src, 'NaN-safe threshold selection'),
]
for ok, name in checks:
    print(f'  {"\u2705" if ok else "\u274c MISSING"} {name}')
    assert ok, f'{name} — push latest code!'

subprocess.run(["pip", "install", "-q", "mlflow", "boto3", "pyarrow", "pandas",
                "numpy", "scikit-learn", "xgboost", "lightgbm"],
               capture_output=True)
print('\u2705 Cloned + verified + deps installed')

In [ ]:
import torch
from google.colab import drive
from pathlib import Path
import shutil

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Runtime > Change runtime type > T4 GPU")
gpu = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
is_modern_gpu = cc[0] >= 8
bs = 4096 if is_modern_gpu else 1024
lr = 1e-3 if is_modern_gpu else 5e-4
amp = 'bfloat16' if is_modern_gpu else 'float16+GradScaler(init_scale=1024)'
print(f'\u2705 {gpu} ({vram:.1f} GB, cc {cc[0]}.{cc[1]})')
print(f'   batch_size={bs}, lr={lr}, {amp}')
if not is_modern_gpu:
    print('   T4/V100 stability mode enabled: batch_size=1024, lr=5e-4')

drive.mount('/content/drive')
dest = '/content/053_dram_memory_yield_mlops/data/preprocessed_full.npz'
Path(dest).parent.mkdir(parents=True, exist_ok=True)

for src_path in ['/content/drive/MyDrive/P053_data/preprocessed_full.npz',
                 '/content/drive/MyDrive/preprocessed_full.npz']:
    if Path(src_path).exists():
        shutil.copy2(src_path, dest)
        mb = Path(dest).stat().st_size / 1e6
        print(f'\u2705 Copied {mb:.0f} MB from {src_path}')
        assert mb > 2000, f'File too small ({mb:.0f} MB)'
        break
else:
    raise FileNotFoundError('preprocessed_full.npz not found on Drive')

In [ ]:
import subprocess, sys, os, time

DIR = '/content/053_dram_memory_yield_mlops'
os.chdir(DIR)

import torch
cc = torch.cuda.get_device_capability(0)[0]
is_modern_gpu = cc >= 8
bs = 4096 if is_modern_gpu else 1024
lr = 1e-3 if is_modern_gpu else 5e-4

# Only 3 epochs for quick test — enough to verify training works
EPOCHS = 3

cmd = [sys.executable, '-u', '-m', 'src.train',
       '--full', '--epochs', str(EPOCHS), '--batch-size', str(bs),
       '--lr', str(lr), '--run-name', 'colab-t4-quick-test', '--context', 'colab']

print(f'Command: {" ".join(cmd)}')
print(f'Epochs: {EPOCHS}')
print(f'Effective config: batch_size={bs}, lr={lr}')
print('=' * 70, flush=True)

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

t0 = time.time()
proc = subprocess.Popen(
    cmd, cwd=DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
retcode = proc.wait()
elapsed = (time.time() - t0) / 60

print(f'\n{"=" * 70}')
if retcode == 0:
    print(f'\u2705 Training complete in {elapsed:.1f} min')
else:
    print(f'\u274c Training FAILED (exit {retcode}) after {elapsed:.1f} min')
print('=' * 70)

In [ ]:
import json
from pathlib import Path

DIR = '/content/053_dram_memory_yield_mlops'
for b in sorted(Path(f'{DIR}/data').glob('benchmark_*.json'),
                key=lambda f: f.stat().st_mtime, reverse=True):
    bm = json.loads(b.read_text())
    print(f'Benchmark: {b.name}')
    print(f'  GPU: {bm.get("gpu_name")} | batch: {bm.get("batch_size")}')
    print(f'  Epochs: {bm.get("epochs_run")} (best: {bm.get("best_epoch")})')
    print(f'  Time: {bm.get("total_train_time_min", 0):.1f} min')
    print(f'  Avg epoch: {bm.get("avg_epoch_time_s", 0):.1f}s')
    print(f'  Peak VRAM: {bm.get("peak_gpu_memory_gb", 0):.1f} GB')
    if 'results' in bm and 'val' in bm['results']:
        v = bm['results']['val']
        print(f'  Val AUC-PR: {v["auc_pr"]:.4f} | F1: {v["f1"]:.4f}')
    break
else:
    print('No benchmark found — training may have failed')

arts = list(Path(f'{DIR}/src/artifacts').glob('*.pt'))
if arts:
    print(f'\n\u2705 Model: {arts[0].name} ({arts[0].stat().st_size/1e6:.1f} MB)')
else:
    print('\n\u26a0\ufe0f No model .pt found')

import shutil
drive_dir = Path('/content/drive/MyDrive/P053_results/training_test')
drive_dir.mkdir(parents=True, exist_ok=True)
for f in Path(f'{DIR}/data').glob('benchmark_*.json'):
    shutil.copy2(f, drive_dir / f.name)
for f in Path(f'{DIR}/src/artifacts').glob('*.pt'):
    shutil.copy2(f, drive_dir / f.name)
print(f'\n\u2705 Results copied to {drive_dir}')